In [1]:
# WHAT: Prepare data for forecasting
# WHY: Machine learning needs structured time-series data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append(r'D:\Vaidehi Study\Regional_Sales_Analysis\04_scripts')
from config import DB_CONFIG, CSV_PATHS

engine = create_engine(f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
df = pd.read_sql("SELECT * FROM v_master_sales_report", con=engine)

print("✅ Data loaded for forecasting")

✅ Data loaded for forecasting


In [3]:
# Preparing Monthly Time Series
# WHAT: Convert daily sales to monthly totals
# WHY: Forecasting works on consistent time periods

# Create month column
df['YearMonth'] = pd.to_datetime(df['OrderDate']).dt.to_period('M')

# Group by month
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum().reset_index()
#Renaming the columns for cleaner names: 'Month' & 'Revenue'
monthly_revenue.columns = ['Month', 'Revenue']
#Creating a new column 'TimeIndex' that assigns an integer index(0,1,2...) to each row, will be used as Variable x down in Linear Regression/Time Series model 
monthly_revenue['TimeIndex'] = range(len(monthly_revenue))

print("=" * 80)
print("HISTORICAL MONTHLY REVENUE (for forecasting)")
print("=" * 80)
print(monthly_revenue)

print(f"\nTotal months of data: {len(monthly_revenue)}")

HISTORICAL MONTHLY REVENUE (for forecasting)
      Month    Revenue  TimeIndex
0   2022-01  1305145.6          0
1   2022-02  1256771.4          1
2   2022-03  1580290.4          2
3   2022-04  1197330.5          3
4   2022-05  1328354.5          4
5   2022-06  1377730.3          5
6   2022-07  1399233.5          6
7   2022-08  1213289.3          7
8   2022-09  1406160.3          8
9   2022-10  1281962.2          9
10  2022-11  1306543.5         10
11  2022-12  1264696.9         11
12  2023-01  1434506.6         12
13  2023-02  1161789.1         13
14  2023-03  1326835.5         14
15  2023-04  1298059.1         15
16  2023-05  1317456.4         16
17  2023-06  1371412.9         17
18  2023-07  1296723.5         18
19  2023-08  1413865.8         19
20  2023-09  1138983.5         20
21  2023-10  1404053.9         21
22  2023-11  1144419.8         22
23  2023-12  1380821.6         23
24  2024-01  1342392.5         24
25  2024-02  1268284.5         25
26  2024-03  1540521.2         26
27 

In [4]:
#Build Linear Regression Model
# WHAT: Create a machine learning model to predict future sales
# WHY: My hybrid Twist
# HOW: Linear Regression = finds the trend line

print("=" * 80)
print("BUILDING LINEAR REGRESSION FORECAST MODEL")
print("=" * 80)

# Prepare data for model
X = monthly_revenue[['TimeIndex']].values  # Input: time period
y = monthly_revenue['Revenue'].values      # Output: revenue

# Create and train model
model = LinearRegression()
model.fit(X, y)

# Make predictions on historical data
y_pred = model.predict(X)

# Calculate model accuracy
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\n📊 MODEL PERFORMANCE:")
print(f"   R² Score: {r2:.4f}  (1.0 = perfect)")
print(f"   RMSE: ${rmse:,.2f}  (prediction error)")
print(f"   Trend: ${model.coef_[0]:,.2f} per month")


if model.coef_[0] > 0:
    print(f"   ✅ Sales trending UP")
else:
    print(f"   ⚠️ Sales trending DOWN")

BUILDING LINEAR REGRESSION FORECAST MODEL

📊 MODEL PERFORMANCE:
   R² Score: 0.0057  (1.0 = perfect)
   RMSE: $106,691.63  (prediction error)
   Trend: $561.61 per month
   ✅ Sales trending UP
